# Polymer Five-Seed Core Study

This notebook runs the polymer core RL methods across five fixed seeds, saves the native per-run bundles/plots, loads the existing baseline MPC save from `Data/`, and exports aggregate slide-ready analysis artifacts.


## Study Scope

- Methods: `horizon_dueling`, `matrix`, `weights`, `residual`, `combined`
- Continuous methods: TD3 only
- Discrete methods: dueling DQN only
- Combined override: dueling DQN horizon + TD3 matrix/weights/residual
- Baseline MPC: imported from the canonical polymer `Data/` save; never rerun here


In [ ]:
from pprint import pprint

from utils.polymer_multiseed_core_study import default_study_config, run_polymer_multiseed_core_study

STUDY = default_study_config()

# Notebook-facing overrides.
METHODS = list(STUDY["methods"])
SEEDS = list(STUDY["seeds"])
SAVE_PDF = bool(STUDY["save_pdf"])
STYLE_PROFILE = STUDY["style_profile"]
FIGURE_PREFIX = STUDY["figure_prefix"]
REWARD_TAIL_EPISODES = int(STUDY["reward_tail_episodes"])
POLYMER_DATA_DIR_OVERRIDE = STUDY["data_dir_override"]
POLYMER_RESULTS_DIR_OVERRIDE = STUDY["results_dir_override"]


In [ ]:
study_config_snapshot = {
    "methods": METHODS,
    "seeds": SEEDS,
    "save_pdf": SAVE_PDF,
    "style_profile": STYLE_PROFILE,
    "figure_prefix": FIGURE_PREFIX,
    "reward_tail_episodes": REWARD_TAIL_EPISODES,
    "data_dir_override": POLYMER_DATA_DIR_OVERRIDE,
    "results_dir_override": POLYMER_RESULTS_DIR_OVERRIDE,
}

pprint(study_config_snapshot)


In [ ]:
from pathlib import Path

CONTINUE_ON_ERROR = True
METHOD_RESULTS = {}

def run_method_cell(method_key):
    result = run_polymer_multiseed_core_study(
        methods=[method_key],
        seeds=SEEDS,
        data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
        results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
        save_pdf=SAVE_PDF,
        style_profile=STYLE_PROFILE,
        figure_prefix=f"{FIGURE_PREFIX}_{method_key}",
        reward_tail_episodes=REWARD_TAIL_EPISODES,
        continue_on_error=CONTINUE_ON_ERROR,
    )
    METHOD_RESULTS[method_key] = result
    failed_runs = result.get("failed_runs", [])
    print({
        "method": method_key,
        "successful_runs": len(result.get("run_records", [])),
        "failed_runs": len(failed_runs),
        "summary_markdown": result.get("summary_markdown"),
    })
    if failed_runs:
        print("Failed runs saved to:", result.get("failed_runs_csv"))
        for failure in failed_runs:
            print(f"seed={failure['seed']} :: {failure['error_type']} :: {failure['error_message']}")
    return result

def show_method_summary(method_key):
    result = METHOD_RESULTS[method_key]
    summary_path = Path(result["summary_markdown"])
    print(summary_path)
    print()
    print(summary_path.read_text(encoding="utf-8"))


In [ ]:
horizon_dueling_result = run_method_cell("horizon_dueling")
horizon_dueling_result


In [ ]:
matrix_result = run_method_cell("matrix")
matrix_result


In [ ]:
weights_result = run_method_cell("weights")
weights_result


In [ ]:
residual_result = run_method_cell("residual")
residual_result


In [ ]:
combined_result = run_method_cell("combined")
combined_result


In [ ]:
for method_key in METHODS:
    if method_key not in METHOD_RESULTS:
        continue
    print(f"===== {method_key} =====")
    show_method_summary(method_key)
    print()


In [ ]:
study_result = run_polymer_multiseed_core_study(
    methods=METHODS,
    seeds=SEEDS,
    data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
    results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
    figure_prefix=FIGURE_PREFIX,
    reward_tail_episodes=REWARD_TAIL_EPISODES,
    continue_on_error=CONTINUE_ON_ERROR,
)

study_result
